# Proyecto Final — Navegación Autónoma CNN Espacio-Temporal
## Universidad Rafael Landívar | Inteligencia Artificial | 2026-1

**Pipeline del notebook:**
1. Verificación del balance del dataset de navegación
2. Visualización del preprocesamiento matricial (NumPy)
3. Arquitectura NavCNN (frame stacking espacio-temporal)
4. Entrenamiento de la CNN de Navegación
5. Evaluación + criterios demo-ready
6. Benchmark de FPS del pipeline completo


In [ ]:
import os, sys, json, time, random
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image as IPImage, display

sys.path.insert(0, os.path.abspath('.'))

from utils import (
    NAV_CLASSES, DATA_NAV_DIR,
    MODEL_NAV_PATH, METRICS_DIR,
    FRAME_STACK, IMG_HEIGHT, IMG_WIDTH, FrameBuffer,
)
from preprocessing import preprocess_frame, to_grayscale, gaussian_blur, sobel_edges, extract_roi
from dataset import NavigationDataset, get_loaders, print_balance_report
from model_nav import build_nav_model
from train import get_device, train as run_train
from evaluate import evaluate

DEVICE = get_device()
print(f'PyTorch : {torch.__version__}')
print(f'Dispositivo : {DEVICE}')
print('Imports OK')


## Bloque 1 — Verificación del Dataset

Antes de entrenar, verifica que el balance es correcto y que ninguna clase está por debajo del 15%.

In [ ]:
print_balance_report()


In [ ]:
def show_class_samples(root_dir, classes, title, n=4):
    rows = len(classes)
    fig, axes = plt.subplots(rows, n, figsize=(n * 2.8, rows * 2.4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for r, cls in enumerate(classes):
        d = os.path.join(root_dir, cls)
        imgs = sorted(f for f in (os.listdir(d) if os.path.isdir(d) else [])
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        sample = random.sample(imgs, min(n, len(imgs))) if imgs else []
        for c in range(n):
            ax = axes[r, c] if rows > 1 else axes[c]
            ax.axis('off')
            if c < len(sample):
                bgr = cv2.imread(os.path.join(d, sample[c]))
                if bgr is not None:
                    ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            if c == 0:
                ax.set_title(cls, fontsize=8, ha='left')
    plt.tight_layout()
    plt.show()

show_class_samples(DATA_NAV_DIR, NAV_CLASSES, "Dataset Navegación — muestras aleatorias")


## Bloque 2 — Pipeline de Preprocesamiento Matricial

Todos los filtros están implementados **manualmente con NumPy** (`_convolve2d` con `sliding_window_view`).
No se usa `cv2.GaussianBlur`, `cv2.Sobel` ni ninguna función de filtrado de OpenCV.

Etapas:
1. BGR → Escala de grises (coeficientes BT.601: 0.299R + 0.587G + 0.114B)
2. Recorte ROI (35% superior descartado)
3. Blur Gaussiano (kernel 5×5, σ=1.0, convolución NumPy)
4. (Opcional) Magnitud Sobel para resaltar bordes de la línea
5. Resize a 64×64 con `cv2.resize` (permitido para operaciones geométricas)


In [ ]:
# Buscar un frame real para visualizar
sample_bgr = None
for cls in NAV_CLASSES:
    d = os.path.join(DATA_NAV_DIR, cls)
    if os.path.isdir(d):
        imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if imgs:
            sample_bgr = cv2.imread(os.path.join(d, imgs[0]))
            break

if sample_bgr is not None:
    gray    = to_grayscale(sample_bgr) / 255.0
    roi     = extract_roi(gray)
    blurred = gaussian_blur(roi, size=5, sigma=1.0)
    edges   = sobel_edges(blurred)
    final   = cv2.resize(blurred, (IMG_WIDTH, IMG_HEIGHT))

    fig, axes = plt.subplots(1, 5, figsize=(19, 3.2))
    data  = [cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB), gray, roi, blurred, edges]
    names = ['1. BGR original', '2. Grises (manual)',
             f'3. ROI recortada\n(top {int(0.35*100)}% descartado)',
             '4. Blur Gaussiano\n(NumPy manual)',
             '5. Bordes Sobel\n(NumPy manual)']
    for ax, img, name in zip(axes, data, names):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(name, fontsize=8.5)
        ax.axis('off')
    plt.suptitle('Pipeline de Preprocesamiento — sin funciones de filtrado de OpenCV', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(METRICS_DIR, 'pipeline_visual.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Guardado: metrics/pipeline_visual.png")
else:
    print("[!] No hay imágenes en el dataset. Ejecuta capture_dataset.py primero.")


In [ ]:
# Visualizar augmentación
from dataset import augment

if sample_bgr is not None:
    base = preprocess_frame(sample_bgr)
    aug_frames = [augment(base.copy(), allow_flip=True) for _ in range(6)]

    fig, axes = plt.subplots(1, 7, figsize=(19, 2.8))
    axes[0].imshow(base, cmap='gray')
    axes[0].set_title('Original preprocesado', fontsize=8)
    axes[0].axis('off')
    for i, af in enumerate(aug_frames):
        axes[i+1].imshow(af, cmap='gray')
        axes[i+1].set_title(f'Augmentado {i+1}', fontsize=8)
        axes[i+1].axis('off')
    plt.suptitle('Data Augmentation — mismo frame, 6 variantes distintas', fontsize=11)
    plt.tight_layout()
    plt.show()


## Bloque 3 — Arquitecturas Neuronales

### Modelo de Navegación: CNN con Frame Stacking

**Estrategia espacio-temporal**: Los últimos `FRAME_STACK=3` frames en escala de grises se apilan como canales.
La entrada tiene forma `(batch, 3, 64, 64)` — equivalente a una imagen RGB pero con información temporal.

**Por qué frame stacking sobre Conv3D o LSTM:**
- Menor latencia: una sola pasada forward de CNN 2D.
- Sin estado oculto entre frames — más simple de depurar.
- El contexto temporal queda implícito en los canales de entrada.

### Modelo de Señales (Bonus): Vision Transformer (ViT)

64 patches de 8×8 + token [CLS] + 4 bloques Transformer + MLP head. Sin pesos preentrenados.


In [ ]:
# Verificar arquitectura y contar parámetros
nav = build_nav_model(str(DEVICE))

total = sum(p.numel() for p in nav.parameters() if p.requires_grad)
print(f'NavCNN  — Parámetros entrenables: {total:,}')
print()

# Prueba de forma de entrada/salida
dummy = torch.zeros(2, FRAME_STACK, IMG_HEIGHT, IMG_WIDTH)
with torch.no_grad():
    out = nav(dummy)
print(f'NavCNN  entrada: {tuple(dummy.shape)} → salida: {tuple(out.shape)}')


## Bloque 4 — Entrenamiento CNN de Navegación

Hiperparámetros: `lr=0.001`, `batch=32`, `epochs=60`, `weight_decay=1e-4`, scheduler `ReduceLROnPlateau`.

> **Nota**: Si el dataset no tiene imágenes aún, este bloque lanzará un error. Captura primero con `capture_dataset.py`.


In [ ]:
# Entrenar modelo de navegación
# Cambiar a resume=True si quieres retomar desde el último checkpoint
run_train('nav', resume=False)


In [ ]:
# Curvas de entrenamiento
path = os.path.join(METRICS_DIR, 'nav_curves.png')
if os.path.exists(path):
    display(IPImage(path, width=800))
else:
    print("Aún no hay curvas guardadas. Entrena primero.")


## Bloque 5 — Evaluación del Modelo de Navegación

Criterios demo-ready:
- Accuracy global ≥ **85%**
- Recall por clase ≥ **75%** en todas las clases


In [ ]:
evaluate()


In [ ]:
# Mostrar matriz de confusión
cm_path = os.path.join(METRICS_DIR, 'nav_eval_confusion.png')
if os.path.exists(cm_path):
    display(IPImage(cm_path, width=600))


In [ ]:
# Leer reporte JSON y mostrar tabla resumen
rpt_path = os.path.join(METRICS_DIR, 'nav_eval_report.json')
if os.path.exists(rpt_path):
    with open(rpt_path) as f:
        rpt = json.load(f)
    print(f"Accuracy: {rpt['accuracy']:.4f}  |  Loss: {rpt['loss']:.4f}\n")
    print(f"{'Clase':<22} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    print('-' * 50)
    for cls, m in rpt['classes'].items():
        flag = ' ← BAJO' if m['recall'] < 0.75 else ''
        print(f"  {cls:<20} {m['precision']:>10.3f} {m['recall']:>8.3f} {m['f1']:>8.3f}{flag}")


## Bloque 6 — Benchmark del Pipeline

Mide los FPS reales del pipeline de inferencia (sin stream de video, solo procesamiento).
**FPS mínimo aceptable para la demo: 8 FPS.**


In [ ]:
from preprocessing import preprocess_frame

nav_model = build_nav_model('cpu')
if os.path.exists(MODEL_NAV_PATH):
    nav_model.load_state_dict(
        torch.load(MODEL_NAV_PATH, map_location='cpu')['model_state'])
    nav_model.eval()
    print('Modelo nav cargado')
else:
    print('[WARN] Modelo no encontrado — usando pesos aleatorios')

# Simular 200 frames
buf   = FrameBuffer(FRAME_STACK)
dummy = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
N     = 200
t0    = time.perf_counter()

with torch.no_grad():
    for _ in range(N):
        proc  = preprocess_frame(dummy)
        buf.push(proc)
        stack = torch.tensor(buf.get_stack()[None], dtype=torch.float32)
        _     = nav_model(stack)

elapsed = time.perf_counter() - t0
fps     = N / elapsed

print(f"\n{'='*45}")
print(f'  Frames procesados : {N}')
print(f'  Tiempo total      : {elapsed:.2f} s')
print(f'  FPS del pipeline  : {fps:.1f}')
print(f"  {'[OK]  Aceptable para demo' if fps >= 8 else '[FAIL] Por debajo del mínimo (8 FPS)'}")
print(f"{'='*45}")
if fps < 8:
    print('\nOptimizaciones disponibles:')
    print('  Bajar resolución a 320x240 en IP Webcam')
    print('  Cambiar hotspot a 5 GHz')
    print('  Cerrar apps pesadas en el Mac durante la demo')


## Resumen de Resultados

| Componente | Criterio | Estado |
|---|---|---|
| NavCNN — Accuracy global | ≥ 85% | *(ver evaluación)* |
| NavCNN — Recall mínimo por clase | ≥ 75% | *(ver evaluación)* |
| FPS pipeline | ≥ 8 FPS | *(ver benchmark)* |

**Siguiente paso**: ejecutar `uv run python main_robot.py --display` con el robot en la pista y el ESP32 conectado al hotspot del Mac.
